# Pipeline de Análise de Produção Acadêmica (Lattes + Scopus + Eventos)

Este notebook é uma **reorganização** do `analyse.ipynb` original. Ele produz
exatamente os mesmos resultados finais (`df_pessoas`, `df_orientacoes`,
`df_artigos_final`, `df_artigos_congresso_final`, `df_alunos` e a carga no
DuckDB), mas com o código agrupado por etapa, comentado e sem fragmentos soltos.

## O que foi limpo em relação ao notebook original

- **Imports**: centralizados todos no topo, em vez de espalhados ao longo do notebook.
- **Células de depuração** (`.head()`, `.info()`, `.value_counts()` soltos, sem
  nenhum tratamento associado): removidas ou incorporadas como prints de
  validação dentro da célula de tratamento correspondente.
- **Duplicação real**: a limpeza de `texto_resumo` em `df_pessoas` existia em
  duas células idênticas seguidas — mantida uma única vez (o efeito é o mesmo).
- **Versões concorrentes do matching de eventos/conferências**: o notebook
  original tinha duas implementações (uma por substring bidirecional, outra
  fuzzy com `rapidfuzz`) escritas em sequência no mesmo DataFrame — a segunda
  sempre sobrescrevia a primeira antes de qualquer uso do resultado. Mantida
  apenas a versão fuzzy (que é, de fato, a que chegava ao resultado final).
- **~430 linhas inteiramente comentadas** (versões antigas e abandonadas da
  lógica de coautoria de alunos): removidas. Não tinham nenhum efeito no
  notebook original (estavam 100% comentadas) e não afetam o resultado.

## Estrutura deste notebook

1. Configuração e imports
2. Extração dos arquivos JSON brutos (Lattes) e consolidação em DataFrames
3. Tratamento de `df_pessoas` (informações pessoais)
4. Tratamento de `df_orientacoes`
5. Tratamento de `df_bib_artigos` (artigos de periódico) e `df_bib_trab_congresso` (trabalhos de congresso)
6. Cruzamento de periódicos com a base de percentil Scopus → `df_artigos_final`
7. Cruzamento de trabalhos de congresso com a base de eventos classificados → `df_artigos_congresso_final`
8. Detecção de coautoria de alunos nas produções
9. Persistência consolidada no banco DuckDB (`pesquisadores.duckdb`)


## 1. Configuração e Imports

Todas as bibliotecas usadas em qualquer etapa do notebook são importadas uma
única vez aqui. No notebook original, vários `import pandas as pd` e
`import numpy as np` se repetiam no meio do código — isso não causa erro
(Python ignora reimportações), mas dificulta saber, de cara, todas as
dependências do projeto.

In [30]:
import json
import re
import unicodedata
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
from rapidfuzz import fuzz

# Caminho onde estão os currículos Lattes (em JSON) dos professores.
# Cada arquivo é o "dump" bruto de um currículo já raspado/processado.
CAMINHO_JSONS_PROFESSORES = 'dados_brutos/professores/ufrj/*.json'

# Caminho da pasta com os currículos Lattes (em JSON) dos alunos, usada
# na Seção 8 para detectar coautoria entre professores e alunos.
CAMINHO_PASTA_ALUNOS = 'dados_brutos/alunos'

# Planilha de percentis Scopus por periódico (usada na Seção 6).
ARQUIVO_PERCENTIL_SCOPUS = 'periodicos_percentil.xlsx'

# Base de eventos/conferências já classificados por estrato (usada na Seção 7).
ARQUIVO_EVENTOS_CLASSIFICADOS = 'eventos_classificados_dois_idiomas.csv'

# Arquivo DuckDB de destino final (usado na Seção 9).
ARQUIVO_DUCKDB_DESTINO = 'pesquisadores_teste.duckdb'

## 2. Extração dos JSONs Brutos e Consolidação em DataFrames

Cada arquivo JSON representa o currículo Lattes de **um professor**, já
estruturado em blocos (`informacoes_pessoais`, `bancas`, `eventos`,
`orientacoes`, `premios_titulos`, `projetos_pesquisa`, `producao_bibliografica`,
`producao_tecnica`, `patentes_registros`).

A estratégia é:

1. Percorrer todos os arquivos JSON da pasta de professores.
2. Para cada bloco de interesse, "achatar" (flatten) a estrutura em uma lista
   de dicionários e marcar cada registro com o `id_lattes` do professor de origem.
3. Acumular essas listas e, ao final, concatenar tudo em um único DataFrame
   por tipo de produção (ex.: todos os artigos de periódico de todos os
   professores em um só `df_bib_artigos`).

Este notebook foca na trilha de **produção bibliográfica de periódicos e
congressos** (linha do meio do diagrama abaixo), mas a extração já deixa
pronto também `df_pessoas`, `df_orientacoes` e os demais DataFrames, caso
sejam necessários em outra análise.

In [31]:
print("Localizando arquivos JSON de professores...")
caminhos_arquivos = glob.glob(CAMINHO_JSONS_PROFESSORES)

if not caminhos_arquivos:
    raise FileNotFoundError(
        f"Nenhum arquivo JSON encontrado em '{CAMINHO_JSONS_PROFESSORES}'. "
        "Verifique se o caminho está correto antes de continuar."
    )

print(f"{len(caminhos_arquivos)} arquivo(s) encontrado(s). Iniciando a extração...")

Localizando arquivos JSON de professores...
39 arquivo(s) encontrado(s). Iniciando a extração...


In [32]:
# ---------------------------------------------------------
# 2.1 Listas de acumulação — uma por tipo de produção/registro
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

In [33]:
# ---------------------------------------------------------
# 2.2 Extração e achatamento (flatten) de cada arquivo JSON
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)

        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            # Sem id_lattes não há como vincular nenhum registro a um professor;
            # o arquivo é descartado.
            continue

        # --- INFORMAÇÕES PESSOAIS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)

        # --- BANCAS (estrutura: {categoria: [itens]}) ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria
                    if 'membros_banca' in df_temp.columns:
                        # Lista de membros é convertida para string para caber em uma célula tabular
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS PARTICIPADOS (estrutura: {categoria: [itens]}) ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES (estrutura: {status: {nivel: [itens]}}) ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS E TÍTULOS (lista simples) ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS DE PESQUISA (lista simples, com campos longos) ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA (estrutura: {chave: [itens]}) ---
        prod_bib = dados.get('producao_bibliografica', {})

        def add_to_list(chave, lista_destino):
            """Extrai uma chave de produção bibliográfica e empilha na lista destino."""
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA (estrutura: {chave: [itens]}) ---
        prod_tec = dados.get('producao_tecnica', {})

        def add_to_list_tec(chave, lista_destino):
            """Extrai uma chave de produção técnica e empilha na lista destino."""
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES E REGISTROS (estrutura: {chave: [itens]}) ---
        patentes = dados.get('patentes_registros', {})

        def add_to_list_pat(chave, lista_destino):
            """Extrai uma chave de patentes/registros e empilha na lista destino."""
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)

print("Extração e achatamento concluídos para todos os arquivos.")

Extração e achatamento concluídos para todos os arquivos.


In [34]:
# ---------------------------------------------------------
# 2.3 Consolidação: cada lista de DataFrames parciais (um por professor)
#     é concatenada em um único DataFrame final por tipo de produção.
# ---------------------------------------------------------
def consolidar(lista):
    """Concatena uma lista de DataFrames parciais; retorna DataFrame vazio se a lista estiver vazia."""
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames de Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames de Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames de Patentes e Registros
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

print("DataFrames consolidados. Resumo de volumes:")
print(f"  Professores (df_pessoas):              {len(df_pessoas)}")
print(f"  Orientações (df_orientacoes):           {len(df_orientacoes)}")
print(f"  Artigos de periódico (df_bib_artigos):  {len(df_bib_artigos)}")
print(f"  Trabalhos de congresso (df_bib_trab_congresso): {len(df_bib_trab_congresso)}")

DataFrames consolidados. Resumo de volumes:
  Professores (df_pessoas):              39
  Orientações (df_orientacoes):           2879
  Artigos de periódico (df_bib_artigos):  1997
  Trabalhos de congresso (df_bib_trab_congresso): 3585


## 3. Tratamento de `df_pessoas` (Informações Pessoais)

Limpeza padrão de cadastro: strings vazias→nulo, datas, remoção de marcador
"*" no rótulo, tipagem da chave primária e do texto de resumo. Estas etapas
são independentes entre si, mas mantidas na mesma ordem do notebook original.

In [35]:
# 3.1 Substitui strings vazias ou só com espaços por NaN (nulo real)
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 3.2 Converte a data de atualização do CV ('15/10/2025') para datetime.
#     errors='coerce' faz datas inválidas virarem nulo em vez de quebrar o script.
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'],
        format='%d/%m/%Y',
        errors='coerce'
    )

# 3.3 Limpeza do campo 'rotulo': remove o asterisco e espaços, e transforma
#     o texto literal "Sem rótulo" em nulo verdadeiro.
if 'rotulo' in df_pessoas.columns:
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

# 3.4 Garante que a chave primária (id_lattes) seja sempre string,
#     evitando inconsistências de tipo em merges/joins posteriores.
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

# 3.5 Remove espaços/quebras de linha nas bordas do texto de resumo do CV.
if 'texto_resumo' in df_pessoas.columns:
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

print("Tratamento de df_pessoas concluído.")
df_pessoas.info()

Tratamento de df_pessoas concluído.
<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lattes              39 non-null     str           
 1   nome_completo          39 non-null     str           
 2   nome_citacoes          39 non-null     str           
 3   sexo                   39 non-null     str           
 4   rotulo                 0 non-null      str           
 5   periodo                0 non-null      str           
 6   bolsa_produtividade    21 non-null     str           
 7   endereco_profissional  38 non-null     str           
 8   atualizacao_cv         39 non-null     datetime64[us]
 9   url                    39 non-null     str           
 10  texto_resumo           39 non-null     str           
dtypes: datetime64[us](1), str(10)
memory usage: 77.1 KB


## 4. Tratamento de `df_orientacoes`

**Atenção à ordem**: a coluna original `titulo` é renomeada para
`titulo_trabalho` *antes* da limpeza de texto em lote, porque a limpeza já
referencia o nome novo (`titulo_trabalho`). Essa dependência existia também
no notebook original (em células separadas) — aqui ela fica explícita.

> **Nota de atenção (herdada do notebook original):** a Seção 9 insere
> `df_orientacoes` no banco esperando uma coluna `ano_inicio`. Essa coluna
> nunca é criada nem renomeada em nenhuma etapa de tratamento — ela só existe
> se o JSON bruto de orientações já trouxer um campo chamado literalmente
> `ano_inicio`. Se o seu JSON de origem usa outro nome para o ano de início da
> orientação, adicione aqui um `rename` (no mesmo padrão da linha abaixo que
> renomeia `titulo` → `titulo_trabalho`) antes de chegar à Seção 9, ou a
> inserção no banco falhará com `KeyError`/coluna inexistente.

In [36]:
# 4.1 Renomeia 'titulo' para 'titulo_trabalho' (nome mais descritivo,
#     usado pela limpeza de texto na sequência e pelo schema do banco).
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

print("Tratamento da tabela de orientações...")

# 4.2 Limpeza de texto: remove espaços duplos/quebras de linha escondidas e
#     preenche vazios com um rótulo explícito em vez de deixá-los como string vazia.
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    df_orientacoes[col] = df_orientacoes[col].replace(
        {'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'}
    )

# 4.3 Conversão segura do ano de conclusão (float -> Int64, que aceita nulos).
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 4.4 Padroniza os valores de 'nivel' para rótulos amigáveis (usados em gráficos).
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4.5 Padroniza os valores de 'status' para rótulos amigáveis.
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("Tratamento concluído. Amostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Tratamento da tabela de orientações...
Tratamento concluído. Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## 5. Tratamento de `df_bib_artigos` (Periódicos) e `df_bib_trab_congresso` (Congressos)

Antes de qualquer cruzamento com bases externas (Scopus / eventos
classificados), os dois DataFrames de produção bibliográfica recebem uma
limpeza básica: nulos reais, padronização de maiúsculas no nome do
periódico/evento, tipagem de ano e remoção de espaços ocultos em texto.

### 5.1 Artigos de Periódico (`df_bib_artigos`)

In [37]:
print("Aplicando tratamentos na tabela 'df_bib_artigos'...")

if not df_bib_artigos.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome da revista em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento exato com a base Scopus na Seção 6)
    if 'revista' in df_bib_artigos.columns:
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64); valores inválidos -> nulo
    if 'ano' in df_bib_artigos.columns:
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 5. Garante tipagem string na chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento de 'df_bib_artigos' concluído.")
display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())

Aplicando tratamentos na tabela 'df_bib_artigos'...
Tratamento de 'df_bib_artigos' concluído.


,ano,revista,doi,issn
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005,1464-5149
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007,0166-218X
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717,0012-365X
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022,0166-218X
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027,1571-0661


### 5.2 Trabalhos de Congresso (`df_bib_trab_congresso`)

In [38]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome do evento em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento com a base de eventos na Seção 7)
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64)
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. Padronização da coluna 'autores': separador único (vírgula),
    #    sem espaços duplicados, e em maiúsculas (facilita buscas futuras,
    #    inclusive a detecção de coautoria de alunos na Seção 8).
    if 'autores' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garante tipagem string na chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento de 'df_bib_trab_congresso' concluído.")
display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento de 'df_bib_trab_congresso' concluído.


,ano,evento,autores
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA..."
1,2021,DIAGRAMS,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO"
2,2017,LATIN AND AMERICAN ALGORITHMS,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,..."
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,"CERIOLI, MA'RCIA, NOBREGA, HUGO , SILVEIRA, GU..."
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,"BARROS, GABRIEL F. , POSNER, DANIEL F. D. , CE..."


## 6. Cruzamento de Periódicos com a Base de Percentil Scopus

Objetivo: para cada artigo de periódico extraído do Lattes (`df_bib_artigos`),
encontrar o registro correspondente na planilha de percentis da Scopus
(`periodicos_percentil.xlsx`), para herdar o **percentil da revista** e a
**área de classificação** — métricas usadas depois para os índices de
avaliação quadrienal.

A estratégia de cruzamento é em camadas, da mais confiável para a mais
flexível, processando apenas o que ainda não teve sucesso na camada anterior:

1. **Match exato por nome da revista** (`revista` == `Title`).
2. **Match exato por ISSN** (eletrônico ou impresso).
3. **Match bidirecional por substring** do nome da revista, só para os
   artigos que sobraram das camadas 1–2 (mais caro computacionalmente,
   por isso roda apenas no restante).
4. O que ainda sobra é marcado como **falha de match** (percentil 0).

Ao final, criamos a coluna booleana `match_adequado` (houve correspondência
com a Scopus ou não) e renomeamos as colunas para o padrão final do projeto.

### 6.1 Preparação das duas bases (limpeza de chaves de cruzamento)

In [ ]:
def formatar_issn(issn):
    """Normaliza um ISSN para 8 dígitos sem hífen, retornando nulo se o valor for vazio/inválido.

    Isso é necessário porque o mesmo periódico pode aparecer com ISSN formatado
    de formas diferentes nas duas bases (com/sem hífen, com/sem zeros à esquerda).
    """
    issn_str = str(issn).replace('-', '').strip()
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    return issn_str.zfill(8)


print("Etapa 1: Carregando e preparando a base completa da Scopus...")
df_scopus_raw = pd.read_excel(ARQUIVO_PERCENTIL_SCOPUS)

# Padroniza a chave de nome (maiúsculas, sem espaços nas bordas)
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()

# Padroniza ambos os ISSNs (eletrônico e impresso) para 8 dígitos
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

# Conjunto de títulos/ISSNs que pertencem a alguma subárea de Computação
# (usado depois para marcar a coluna 'Computation Area')
mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                    set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Um mesmo periódico pode aparecer várias vezes na planilha (uma linha por
# subárea ASJC). Mantemos apenas o percentil mais alto de cada título, que é
# o cenário mais favorável ao pesquisador.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

print("Etapa 2: Preparando a base de artigos do Lattes...")
df_bib_artigos['revista'] = df_bib_artigos['revista'].astype(str).str.upper().str.strip()
df_bib_artigos['issn'] = df_bib_artigos['issn'].apply(formatar_issn)
df_bib_artigos['doi'] = (
    df_bib_artigos['doi']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': pd.NA, 'none': pd.NA, '': pd.NA})
)

print("Preparação concluída.")

Etapa 1: Carregando e preparando a base completa da Scopus...


### 6.2 Camadas 1 e 2 — Match exato (nome da revista e ISSN)

> **Nota de fidelidade ao comportamento original:** quando tanto `issn` (Lattes)
> quanto `E-ISSN`/`Print ISSN` (Scopus) são nulos, o `pd.merge` do pandas trata
> os dois nulos como iguais e gera um match "por ISSN" mesmo sem nenhum ISSN
> de fato existir nos dois lados. Esse comportamento já existia no notebook
> original (não é uma falha introduzida nesta reorganização) — mantido aqui
> propositalmente para que os resultados finais sejam idênticos. Se desejar
> corrigir esse ponto no futuro, basta filtrar `issn.notna()` antes de cada
> merge por ISSN.

In [ ]:
print("Etapa 3: Realizando o cruzamento exato (ISSN e Nome Exato)...")

colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

# Três tentativas de match exato, cada uma usando uma chave diferente
df_match_nome = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')
df_match_e_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')
df_match_print_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

# Une os três resultados e remove duplicatas (um mesmo artigo pode ter
# batido em mais de uma das três tentativas)
df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

# Marca se o periódico pertence à área de Computação
df_sucessos['Computation Area'] = (
    df_sucessos['Title'].isin(titulos_computacao) |
    df_sucessos['E-ISSN'].isin(issns_computacao) |
    df_sucessos['Print ISSN'].isin(issns_computacao)
)

print(f"Matches exatos encontrados: {len(df_sucessos)}")

Etapa 3: Realizando o cruzamento exato (ISSN e Nome Exato)...
Matches exatos encontrados: 1668


### 6.3 Camada 3 — Match bidirecional por substring (apenas no restante)

In [ ]:
print("Etapa 4: Busca bidirecional (nome contido) para os artigos restantes...")

# Identifica quais artigos ainda não tiveram sucesso nas camadas anteriores
artigos_com_match_exato = set(df_sucessos['titulo'] + df_sucessos['id_lattes'])
df_restante = df_bib_artigos[
    ~(df_bib_artigos['titulo'] + df_bib_artigos['id_lattes']).isin(artigos_com_match_exato)
].copy()

# Ordena a lista de periódicos Scopus do nome mais longo para o mais curto.
# Isso evita que um nome curto "roube" o match de um nome mais específico/longo
# que também o contivesse como substring (ex.: "REVISTA DE FÍSICA" não deve
# bater antes de "REVISTA DE FÍSICA APLICADA" quando o segundo é o correto).
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')


def busca_bidirecional_revista(revista_lattes):
    """Procura, na lista Scopus, um título que contenha (ou esteja contido em) o nome da revista do Lattes."""
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus
    return None


# Aplica a busca apenas no restante (economiza processamento)
resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]

    df_match_parcial['Computation Area'] = (
        df_match_parcial['Title'].isin(titulos_computacao) |
        df_match_parcial['E-ISSN'].isin(issns_computacao) |
        df_match_parcial['Print ISSN'].isin(issns_computacao)
    )

    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

print(f"Total de sucessos após a busca bidirecional: {len(df_sucessos)}")

Etapa 4: Busca bidirecional (nome contido) para os artigos restantes...
Total de sucessos após a busca bidirecional: 1922


### 6.4 Falhas de match e consolidação final

In [ ]:
print("Etapa 5: Isolando e tratando as falhas definitivas...")

# Quem não bateu em nenhuma das camadas anteriores recebe campos vazios/zerados
df_falhas = df_restante[~mask_encontrados].copy()
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA
df_falhas['Computation Area'] = False

print("Etapa 6: Consolidando a tabela final...")
df_artigos_final = pd.concat([df_sucessos, df_falhas], ignore_index=True)

df_artigos_final['Percentile'] = df_artigos_final['Percentile'].astype(int)
df_artigos_final['Computation Area'] = df_artigos_final['Computation Area'].astype(bool)

print(f"Tabela final construída. Total de linhas: {len(df_artigos_final)}")

colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'doi', 'autores',
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN',
    'Computation Area'
]
df_artigos_final = df_artigos_final[colunas_finais]

display(df_artigos_final.sample(min(10, len(df_artigos_final))))

Etapa 5: Isolando e tratando as falhas definitivas...
Etapa 6: Consolidando a tabela final...
Tabela final construída. Total de linhas: 1990


,id_lattes,titulo,revista,ano,doi,autores,Scopus Source ID,Title,Percentile,Scopus ASJC Code (Sub-subject Area),Scopus Sub-Subject Area,E-ISSN,Print ISSN,Computation Area
450,3957046121364560,Biclique-colouring verification complexity and...,DISCRETE APPLIED MATHEMATICS,2015,http://dx.doi.org/10.1016/j.dam.2014.05.001,"MACÊDO FILHO, H.B. ; DANTAS, S. ; MACHADO, R.C...",25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,0166218X,False
129,2002515486942024,On the resilience of canonical reducible permu...,DISCRETE APPLIED MATHEMATICS,2018,NaN,"BENTO, L. M. S. ; BOCARDO, D. ; MACHADO, R. C....",25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,0166218X,False
1713,5349830056087028,Transforming state regular schools into techni...,JOURNAL OF SMART CITIES AND SOCIETY,2025,http://dx.doi.org/10.1177/27723577251388359,"MONTEIRO, A. ; VIEIRA, E. F. ; GUIMARAES, V. A...",21101068929,SMART CITIES,98,3322,Urban Studies,26246511,NaN,False
953,8130520066599912,Hindsight to foresight: an AI-powered analysis...,EUROPEAN JOURNAL OF FUTURES RESEARCH,2024,http://dx.doi.org/10.1186/s40309-024-00233-5,"BARBOSA, CARLOS EDUARDO ; LYRA, ALAN ; DE SOUZ...",21100850750,EUROPEAN JOURNAL OF FUTURES RESEARCH,94,3301,Social Sciences (miscellaneous),21952248,21954194,False
1688,2002515486942024,On the Helly number in the P3 and related conv...,MATEMATICA CONTEMPORANEA,2019,NaN,"CARVALHO JR., M. T. ; DANTAS, S. ; DOURADO, M....",10300153362,CONTEMPORANEA,45,1202,History,NaN,11273070,False
712,7541486051032916,Cross versus within-company cost estimation st...,IEEE TRANSACTIONS ON SOFTWARE ENGINEERING,2007,http://dx.doi.org/10.1109/tse.2007.1001,"Kitchenham, B. ; Mendes, E. ; TRAVASSOS, G. H.",18711,IEEE TRANSACTIONS ON SOFTWARE ENGINEERING,92,1712,Software,NaN,00985589,False
160,2002515486942024,"Powers of Cycles, Powers of Paths, and Distanc...",DISCRETE APPLIED MATHEMATICS,2011,http://dx.doi.org/10.1016/j.dam.2010.03.012,"LIN, Min C ; Rautenbach, D ; SOULIGNAC, F. ; S...",25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,0166218X,False
164,2002515486942024,Partitioning a Graph into Convex Sets,DISCRETE MATHEMATICS,2011,http://dx.doi.org/10.1016/j.disc.2011.05.023,"ARTIGAS, D. ; DANTAS, S. ; DOURADO, M. C. ; SZ...",25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,0012365X,True
838,4436183480921146,Localização Ótima de Torres de Transmissão Uti...,PESQUISA OPERACIONAL,1979,NaN,"MACULAN FILHO, N.; FIGUEIREDO, J. N. ; GONZAGA...",5000153602,PESQUISA OPERACIONAL,20,1803,Management Science and Operations Research,NaN,01017438,False
242,9358511568098561,Survivability analysis of power distribution i...,PERFORMANCE EVALUATION REVIEW,2012,http://dx.doi.org/10.1145/2425248.2425260,"Menasché, Daniel S. ; LEÃO, R. M. M. ; de Souz...",26742,PERFORMANCE EVALUATION REVIEW,25,1705,Computer Networks and Communications,NaN,01635999,True


### 6.5 Coluna `match_adequado` e renomeação final das colunas

In [ ]:
# 'match_adequado' indica se o artigo encontrou correspondência válida na Scopus
# (ou seja, se possui um 'Scopus Source ID' preenchido).
df_artigos_final['match_adequado'] = df_artigos_final['Scopus Source ID'].notna()

colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'doi', 'autores', 'match_adequado',
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN',
    'Computation Area'
]
df_artigos_final = df_artigos_final[colunas_finais]

print("Coluna 'match_adequado' adicionada.")
display(df_artigos_final[['titulo', 'revista', 'match_adequado', 'Percentile']].sample(min(10, len(df_artigos_final))))

Coluna 'match_adequado' adicionada.


,titulo,revista,match_adequado,Percentile
1195,Development of transient fault management meth...,REVISTA IEEE AMÉRICA LATINA,True,62
534,Age distribution of multiple functionally rele...,FRONTIERS IN IMMUNOLOGY,True,85
1237,Le Problème de Steiner sur un Graphe Orienté: ...,COMPUTATIONAL & APPLIED MATHEMATICS,True,79
4,"L(2,1)-labelling of graphs with few P4?s",DISCRETE OPTIMIZATION,True,58
1133,Uncertainty Quantification in Computational Pr...,INTERNATIONAL JOURNAL FOR UNCERTAINTY QUANTIFI...,True,91
291,Exact approaches to solve the Transmission Exp...,ELECTRIC POWER SYSTEMS RESEARCH,True,84
1698,Sobre o Problema de Automação do Projeto de Pl...,DATA NEWS,True,69
653,Two novel evolutionary formulations of the gra...,JOURNAL OF COMBINATORIAL OPTIMIZATION,True,79
1444,An Online Learning Approach: A Methodology for...,NEURAL COMPUTING & APPLICATIONS,True,90
776,The discretizable molecular distance geometry ...,COMPUTATIONAL OPTIMIZATION AND APPLICATIONS,True,72


In [ ]:
print("Renomeando as colunas para o padrão final do projeto...")

mapeamento_colunas = {
    'titulo': 'titulo_artigo',
    'revista': 'titulo_revista_lattes',
    'ano': 'ano_pub',
    'Scopus Source ID': 'id_scopus',
    'Title': 'titulo_revista_scopus',
    'Percentile': 'maior_percentil',
    'Scopus ASJC Code (Sub-subject Area)': 'codigo_area_maior_percentil',
    'Scopus Sub-Subject Area': 'area_maior_percentil',
    'E-ISSN': 'issn',
    'Computation Area': 'computation_area'
}
df_artigos_final.rename(columns=mapeamento_colunas, inplace=True)

print("Colunas renomeadas. Estrutura final de df_artigos_final:")
df_artigos_final.info()

Renomeando as colunas para o padrão final do projeto...
Colunas renomeadas. Estrutura final de df_artigos_final:
<class 'pandas.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    1990 non-null   str   
 1   titulo_artigo                1990 non-null   str   
 2   titulo_revista_lattes        1990 non-null   str   
 3   ano_pub                      1990 non-null   Int64 
 4   doi                          1512 non-null   str   
 5   autores                      1990 non-null   str   
 6   match_adequado               1990 non-null   bool  
 7   id_scopus                    1922 non-null   object
 8   titulo_revista_scopus        1922 non-null   object
 9   maior_percentil              1990 non-null   int64 
 10  codigo_area_maior_percentil  1922 non-null   object
 11  area_maior_percentil         1922 non-null   

## 7. Cruzamento de Trabalhos de Congresso com a Base de Eventos Classificados

Objetivo: para cada trabalho de congresso extraído do Lattes
(`df_bib_trab_congresso`), encontrar o evento correspondente na planilha de
eventos já classificados por estrato (`eventos_classificados_dois_idiomas.csv`),
de onde vem o **estrato CAPES-like** (A1 a A8) do evento.

Diferente do cruzamento de periódicos (Seção 6), aqui usamos uma única
estratégia: comparação por similaridade textual (*fuzzy matching*) com a
biblioteca `rapidfuzz`, que tolera pequenas diferenças de grafia entre o nome
do evento como o pesquisador digitou no Lattes e o nome oficial na base de
referência.

> **Nota sobre a reorganização:** o notebook original continha duas
> implementações desse cruzamento em células separadas — uma por correspondência
> exata/substring bidirecional, e esta aqui, por similaridade fuzzy. Como a
> segunda sempre era executada depois e sobrescrevia o resultado da primeira
> antes de qualquer uso, apenas a versão fuzzy (que é a que efetivamente
> compunha o resultado final) foi mantida nesta reorganização.

### 7.1 Carga e padronização da base de eventos classificados (Google/CAPES)

In [ ]:
print("Carregando a base de eventos classificados...")
df_google_raw = pd.read_csv(ARQUIVO_EVENTOS_CLASSIFICADOS)

print("Distribuição original de estratos:")
display(df_google_raw['Estrato'].value_counts())

Carregando a base de eventos classificados...
Distribuição original de estratos:


Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [ ]:
# A base de eventos usa rótulos B1-B4 para os estratos mais baixos; o projeto
# usa a faixa estendida A1-A8, então B1-B4 são remapeados para A5-A8.
print("Remapeando estratos B1-B4 -> A5-A8...")

mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

print("Nova distribuição de estratos:")
display(df_google_raw['Estrato'].value_counts())

# Padroniza o nome do evento em maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

Remapeando estratos B1-B4 -> A5-A8...
Nova distribuição de estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

### 7.2 Normalização de texto (remoção de acentos, maiúsculas, espaços)

In [ ]:
print("Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...")


def limpar_texto(serie):
    """Remove acentos, converte para maiúsculas, colapsa espaços múltiplos e tira espaços nas bordas."""
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip())


df_bib_trab_congresso['evento_limpo'] = limpar_texto(df_bib_trab_congresso['evento'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês'])
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# Para a busca por substring/sigla funcionar bem nos dois idiomas, ordenamos a
# base de eventos pelo maior nome disponível entre PT e EN. Isso evita que um
# nome curto "roube" o match de um nome mais longo e específico.
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

# Lista de dicionários — acesso mais rápido do que iterrows() em um DataFrame
lista_google = df_google_raw.to_dict('records')

print("Normalização concluída.")

Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...
Normalização concluída.


### 7.3 Função de match fuzzy (sigla exata > similaridade textual PT/EN)

A função tenta, em ordem de confiança:

1. **Sigla exata**, isolada por limites de palavra (regex `\b`) — score 100,
   match imediato (uma sigla como "ICSE" cravada no meio do texto é um sinal
   muito forte e praticamente sem ambiguidade).
2. **Similaridade fuzzy** (`token_set_ratio`, que ignora ordem e repetição de
   palavras) contra o nome do evento em português e em inglês, mantendo o
   maior score entre os dois.

Só é considerado match válido se o melhor score atingir o limiar de corte
(`LIMIAR_CORTE_FUZZY = 95`, em uma escala de 0 a 100) — um valor alto,
deliberadamente rigoroso, para evitar falsos positivos entre eventos
parecidos mas distintos.

In [ ]:
LIMIAR_CORTE_FUZZY = 95  # Escala 0-100; valor alto para evitar falsos positivos


def encontrar_melhor_match_fuzzy(evento_lattes):
    """Procura, na base de eventos classificados, o melhor match fuzzy para um nome de evento do Lattes.

    Retorna uma tupla: (sigla, nome_do_evento_padronizado, estrato, tipo_match, score_confianca).
    Quando não há match acima do limiar, retorna estrato 'A8' (pior classificação) e tipo 'Sem Match'.
    """
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # --- Tentativa 1: sigla exata isolada por limites de palavra ---
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # --- Tentativa 2: similaridade fuzzy (token_set_ratio) em PT e EN ---
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']

        score_pt = 0
        score_en = 0

        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)

        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        score_atual_max = max(score_pt, score_en)

        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # --- Decisão final: o melhor score supera o limiar de segurança? ---
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'],
            melhor_google_match['Nome do evento'],
            melhor_google_match['Estrato'],
            tipo_do_melhor_match,
            maior_score_encontrado
        )

    # Score insuficiente: rejeita o match para evitar falso positivo
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

### 7.4 Aplicação do match e consolidação de `df_artigos_congresso_final`

In [ ]:
print("Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...")

resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

df_artigos_congresso_final = df_bib_trab_congresso.copy()
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]
df_artigos_congresso_final['score_confianca'] = [res[4] for res in resultados]

# A coluna auxiliar de texto normalizado não é mais necessária
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# --- Relatório-resumo do cruzamento ---
total_originais = len(df_artigos_congresso_final)
qtd_sigla = (df_artigos_congresso_final['tipo_match'] == 'Por Sigla Exata').sum()
qtd_fuzzy_pt = (df_artigos_congresso_final['tipo_match'] == 'Fuzzy Nome PT').sum()
qtd_fuzzy_en = (df_artigos_congresso_final['tipo_match'] == 'Fuzzy Nome EN').sum()
qtd_falhas = (df_artigos_congresso_final['tipo_match'] == 'Sem Match').sum()

print("\n--- Relatório de Cruzamento de Eventos ---")
print(f"Total de trabalhos de congresso (Lattes): {total_originais}")
print(f"Match por Sigla Exata: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"Match Fuzzy (Nome PT):  {qtd_fuzzy_pt} ({round((qtd_fuzzy_pt/total_originais)*100, 1)}%)")
print(f"Match Fuzzy (Nome EN):  {qtd_fuzzy_en} ({round((qtd_fuzzy_en/total_originais)*100, 1)}%)")
print(f"Sem Match:              {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

display(
    df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] != 'Sem Match']
    [['evento', 'Nome do evento', 'Estrato', 'tipo_match', 'score_confianca']]
    .sample(min(5, total_originais))
)

Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...

--- Relatório de Cruzamento de Eventos ---
Total de trabalhos de congresso (Lattes): 3585
Match por Sigla Exata: 1225 (34.2%)
Match Fuzzy (Nome PT):  110 (3.1%)
Match Fuzzy (Nome EN):  763 (21.3%)
Sem Match:              1487 (41.5%)


,evento,Nome do evento,Estrato,tipo_match,score_confianca
2738,HPDGRID 2006 - INTERNATIONAL WORKSHOP ON HIGH-...,CONFERENCIA INTERNACIONAL SOBRE CIENCIA DE DAD...,A1,Por Sigla Exata,100.0
3071,IV WORKSHOP-SCHOOL IN QUANTUM COMPUTATION AND ...,CONFERENCIA INTERNACIONAL SOBRE VISUALIZACAO D...,A4,Por Sigla Exata,100.0
2452,ECSCW - THE 22ND EUROPEAN CONFERENCE ON COMPUT...,CONFERENCIA EUROPEIA SOBRE TRABALHO COOPERATIV...,A4,Por Sigla Exata,100.0
2221,GEOINFO - BRAZILIAN SYMPOSIUM ON GEOINFORMATICS,SIMPOSIO BRASILEIRO DE GEOINFORMATICA,A6,Por Sigla Exata,100.0
2485,2021 IEEE - 24TH INTERNATIONAL CONFERENCE ON C...,CONFERENCIA INTERNACIONAL IEEE SOBRE TRABALHO ...,A3,Fuzzy Nome EN,100.0


### 7.5 Auditoria da "zona crítica" de confiança

Matches com score entre o limiar mínimo (95) e quase-perfeito (99) merecem
uma segunda olhada manual antes de confiar 100% no resultado — scores 100
geralmente vêm de match por sigla exata ou nome idêntico, enquanto a faixa
95–99 indica nomes muito parecidos, mas não idênticos (risco de homônimos).

In [ ]:
limiar_inferior = 95
limiar_superior = 99

df_zona_critica = df_artigos_congresso_final[
    (df_artigos_congresso_final['score_confianca'] >= limiar_inferior) &
    (df_artigos_congresso_final['score_confianca'] <= limiar_superior)
].copy()

df_zona_critica = df_zona_critica.sort_values(by='score_confianca', ascending=True)

colunas_para_auditoria = ['evento', 'Nome do evento', 'Sigla', 'Estrato', 'tipo_match', 'score_confianca']
tabela_auditoria = df_zona_critica[colunas_para_auditoria]

print(f"Encontrados {len(tabela_auditoria)} registros na zona crítica (score {limiar_inferior} a {limiar_superior}).")
print("Recomenda-se leitura atenta para garantir que não há homônimos.\n")

display(
    tabela_auditoria.style.background_gradient(
        subset=['score_confianca'],
        cmap='YlOrRd_r',
        vmin=limiar_inferior,
        vmax=limiar_superior
    )
)

Encontrados 123 registros na zona crítica (score 95 a 99).
Recomenda-se leitura atenta para garantir que não há homônimos.



,evento,Nome do evento,Sigla,Estrato,tipo_match,score_confianca
414,XIX SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1073,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1074,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1080,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1081,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1082,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1083,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1062,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1071,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1347,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951


### 7.6 Renomeação final das colunas e remoção de colunas órfãs

In [ ]:
print("Padronizando os nomes das colunas de eventos...")

mapeamento_colunas_eventos = {
    'titulo': 'titulo_artigo',
    'evento': 'titulo_evento_lattes',
    'Sigla': 'sigla_evento_google',
    'Nome do evento': 'titulo_evento_google',
    'Estrato': 'estrato'
}
df_artigos_congresso_final.rename(columns=mapeamento_colunas_eventos, inplace=True)

# 'cidade' e 'isbn' não fazem parte do schema final do banco; removidas com
# errors='ignore' para a célula ser segura mesmo se já tiverem sido removidas antes.
print("Removendo as colunas 'cidade' e 'isbn' (não usadas no schema final)...")
df_artigos_congresso_final.drop(columns=['cidade', 'isbn'], inplace=True, errors='ignore')

print("Estrutura final de df_artigos_congresso_final:")
df_artigos_congresso_final.info()

Padronizando os nomes das colunas de eventos...
Removendo as colunas 'cidade' e 'isbn' (não usadas no schema final)...
Estrutura final de df_artigos_congresso_final:
<class 'pandas.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   titulo_artigo         3585 non-null   str    
 1   ano                   3585 non-null   Int64  
 2   doi                   981 non-null    str    
 3   autores               3585 non-null   str    
 4   titulo_evento_lattes  3573 non-null   str    
 5   paginas               2505 non-null   str    
 6   id_lattes             3585 non-null   str    
 7   sigla_evento_google   2098 non-null   str    
 8   titulo_evento_google  2098 non-null   str    
 9   estrato               3585 non-null   str    
 10  tipo_match            3585 non-null   str    
 11  score_confianca       3585 non-null   float64
dtypes: Int64(1), float6

## 8. Detecção de Coautoria de Alunos nas Produções

Objetivo: marcar, em `df_artigos_final` e `df_artigos_congresso_final`, quais
publicações tiveram a **coautoria de algum aluno** orientado (cadastrado em
`dados_brutos/alunos`).

O desafio é que o nome de um autor pode aparecer de muitas formas diferentes
na string de autores de uma publicação (`"SILVA, J. V."`,
`"J. V. SILVA"`, `"SILVA, JOÃO VÍTOR"`, etc.), e nem sempre a abreviação
"oficial" do Lattes é a que o sistema de indexação registrou. A estratégia é:

1. **Gerar exaustivamente** todas as variações plausíveis do nome de cada
   aluno (sobrenome + iniciais, sobrenome do meio, nome completo, ordem
   invertida etc.).
2. Para cada publicação, normalizar a string de autores do mesmo jeito e
   verificar se **alguma** dessas variações aparece como substring isolada
   (entre espaços, para não casar com parte de outra palavra).

### 8.1 Normalização de texto e geração de variações de nome

In [ ]:
def normalizar_texto(valor):
    """Remove acentos, força maiúsculas e reduz qualquer caractere não alfanumérico a um único espaço."""
    if pd.isna(valor):
        return ""

    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('utf-8')
    texto = texto.upper()
    texto = re.sub(r'[^A-Z0-9]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def gerar_todas_abreviacoes(nome_completo_norm):
    """Gera o conjunto de variações plausíveis de citação acadêmica de um nome completo normalizado.

    Cobre os padrões mais comuns na autoria brasileira: uso do sobrenome
    materno/intermediário como "sobrenome de citação", sobrenomes compostos,
    nomes de batismo duplos e a inversão Sobrenome-Nome / Nome-Sobrenome.
    """
    preposicoes = {'DE', 'DA', 'DO', 'DAS', 'DOS', 'E'}
    partes_originais = nome_completo_norm.split()
    partes_uteis = [p for p in partes_originais if p not in preposicoes]

    if len(partes_uteis) < 2:
        return {nome_completo_norm}

    variacoes = set([nome_completo_norm])

    # --- Define os possíveis blocos de "sobrenome de citação" ---
    sobrenomes_alvo = [partes_uteis[-1]]  # último nome (ex.: "CARNEIRO")

    # Qualquer nome do meio também pode ser o sobrenome de citação
    # (cobre casos como assinar pelo sobrenome materno)
    for i in range(1, len(partes_uteis) - 1):
        sobrenomes_alvo.append(partes_uteis[i])

    # Combinação composta dos dois últimos nomes (ex.: "DIAS CARNEIRO")
    if len(partes_uteis) >= 3:
        sobrenomes_alvo.append(f"{partes_uteis[-2]} {partes_uteis[-1]}")

    # Com a preposição original, se existir (ex.: "DE CARNEIRO")
    if len(partes_originais) >= 2 and partes_originais[-2] in preposicoes:
        sobrenomes_alvo.append(f"{partes_originais[-2]} {partes_originais[-1]}")

    # --- Combina cada sobrenome candidato com as iniciais do nome restante ---
    for sobrenome in set(sobrenomes_alvo):
        sobrenome_partes = sobrenome.split()
        resto = [p for p in partes_uteis if p not in sobrenome_partes]

        if not resto:
            continue

        iniciais = [p[0] for p in resto]
        primeiro_nome = resto[0]
        inicial_primeira = iniciais[0]

        iniciais_com_espaco = " ".join(iniciais)
        iniciais_sem_espaco = "".join(iniciais)
        duas_iniciais = f"{iniciais[0]} {iniciais[1]}" if len(iniciais) > 1 else inicial_primeira

        primeiro_mais_iniciais = primeiro_nome
        if len(iniciais) > 1:
            primeiro_mais_iniciais += " " + " ".join(iniciais[1:])

        # Possíveis blocos do "nome de batismo" (a parte antes do sobrenome)
        blocos_nome = [
            iniciais_com_espaco,       # ex.: "J V D C"
            iniciais_sem_espaco,       # ex.: "JVDC"
            inicial_primeira,          # ex.: "J"
            duas_iniciais,             # ex.: "J V"
            primeiro_nome,             # ex.: "JOAO"
            primeiro_mais_iniciais,    # ex.: "JOAO V D C"
            " ".join(resto)            # ex.: "JOAO VITOR DIAS CARNEIRO"
        ]

        # Permutação da ordem: Sobrenome-Nome e Nome-Sobrenome
        for bloco in set(blocos_nome):
            variacoes.add(f"{sobrenome} {bloco}")
            variacoes.add(f"{bloco} {sobrenome}")

    return variacoes

### 8.2 Extração dos alunos a partir dos JSONs brutos

In [ ]:
def extrair_alunos(diretorio_alunos):
    """Lê os JSONs brutos dos alunos e monta um DataFrame com todas as variações de nome geradas."""
    registros = []
    pasta = Path(diretorio_alunos)

    if not pasta.exists() or not pasta.is_dir():
        print(f"AVISO: diretório de alunos não encontrado: {diretorio_alunos}")
        return pd.DataFrame()

    for arquivo_json in sorted(pasta.glob('*.json')):
        try:
            with open(arquivo_json, 'r', encoding='utf-8') as f:
                dados_aluno = json.load(f)

            info = dados_aluno.get('informacoes_pessoais', {})
            id_lattes = info.get('id_lattes')
            nome_completo = info.get('nome_completo', '')
            nome_citacoes = info.get('nome_citacoes', '')

            if not id_lattes or not nome_completo:
                continue

            # Variações de citação que o próprio Lattes do aluno já declara
            citacoes = [item.strip() for item in str(nome_citacoes).split(';') if item.strip()]

            nome_comp_norm = normalizar_texto(nome_completo)
            todas_permutacoes = gerar_todas_abreviacoes(nome_comp_norm)

            variacoes_normalizadas = list(todas_permutacoes)
            for variacao in citacoes:
                nome_norm_citacao = normalizar_texto(variacao)
                if nome_norm_citacao and nome_norm_citacao not in variacoes_normalizadas:
                    variacoes_normalizadas.append(nome_norm_citacao)

            registros.append({
                'id_lattes': str(id_lattes),
                'nome_completo': str(nome_completo).strip(),
                'nome_citacoes': str(nome_citacoes).strip(),
                'nome_completo_normalizado': nome_comp_norm,
                'nome_citacoes_normalizadas': ' | '.join(variacoes_normalizadas),
                'variacoes_coautoria': ' | '.join(variacoes_normalizadas),
            })
        except (json.JSONDecodeError, OSError) as erro:
            print(f"Erro ao ler {arquivo_json}: {erro}")

    df_alunos_local = pd.DataFrame(registros)
    if not df_alunos_local.empty:
        df_alunos_local = df_alunos_local.drop_duplicates(subset=['id_lattes']).copy()

    return df_alunos_local


print("Extraindo dados de alunos...")
df_alunos = extrair_alunos(CAMINHO_PASTA_ALUNOS)
print(f"Total de alunos carregados: {len(df_alunos)}")
display(df_alunos.head())

Extraindo dados de alunos...
Total de alunos carregados: 297


,id_lattes,nome_completo,nome_citacoes,nome_completo_normalizado,nome_citacoes_normalizadas,variacoes_coautoria
0,1766965412894981,João Luís da Silva Guio Soares,"SOARES, J. L. S. G.;GUIO, J. L.",JOAO LUIS DA SILVA GUIO SOARES,GUIO JOAO | JOAO L S GUIO SOARES | J SILVA | G...,GUIO JOAO | JOAO L S GUIO SOARES | J SILVA | G...
1,0352188533423371,David Ventura Cardoso,"CARDOSO, D. V.",DAVID VENTURA CARDOSO,DAVID C VENTURA | CARDOSO D | VENTURA CARDOSO ...,DAVID C VENTURA | CARDOSO D | VENTURA CARDOSO ...
2,8773283315440616,Ana Clara Correa da Silva,"SILVA, A. C. C.",ANA CLARA CORREA DA SILVA,CORREA SILVA ANA | A CORREA | CORREA SILVA ANA...,CORREA SILVA ANA | A CORREA | CORREA SILVA ANA...
3,6910314996365495,Fabio Luiz Silva Nogueira,"NOGUEIRA, F. L. S.",FABIO LUIZ SILVA NOGUEIRA,FABIO NOGUEIRA | SILVA NOGUEIRA FABIO L | FL S...,FABIO NOGUEIRA | SILVA NOGUEIRA FABIO L | FL S...
4,2636706873331793,Felipe Bevilaqua Foldes Guimarães,"GUIMARÃES, F. B. F.;GUIMARÃES, FELIPE BEVILAQU...",FELIPE BEVILAQUA FOLDES GUIMARAES,FOLDES GUIMARAES F | FBF GUIMARAES | GUIMARAES...,FOLDES GUIMARAES F | FBF GUIMARAES | GUIMARAES...


### 8.3 Verificação de coautoria nas duas tabelas de produção

In [ ]:
def montar_lista_variacoes(df_alunos_local):
    """Transforma a coluna 'variacoes_coautoria' de cada aluno em um conjunto de strings, agrupados por aluno."""
    variacoes = []
    if df_alunos_local.empty:
        return variacoes

    for _, linha in df_alunos_local.iterrows():
        nomes = [item.strip() for item in str(linha.get('variacoes_coautoria', '')).split('|') if item.strip()]
        if nomes:
            variacoes.append(set(nomes))

    return variacoes


def tem_coautoria_aluno(autores, variacoes_alunos):
    """Verifica se a string de autores de uma publicação contém alguma variação de nome de algum aluno.

    A comparação usa espaços como delimitadores nas duas pontas para evitar
    que uma variação curta (ex.: uma única inicial) seja encontrada como
    substring de outra palavra sem relação nenhuma.
    """
    if pd.isna(autores) or not str(autores).strip() or not variacoes_alunos:
        return False

    autores_normalizados = f" {normalizar_texto(autores)} "

    for variacoes in variacoes_alunos:
        for nome_normalizado in variacoes:
            if f" {nome_normalizado} " in autores_normalizados:
                return True

    return False


print("Marcando coautoria de alunos em df_artigos_final e df_artigos_congresso_final...")
variacoes_alunos = montar_lista_variacoes(df_alunos)

for df_prod in [df_artigos_final, df_artigos_congresso_final]:
    if 'autores' in df_prod.columns:
        df_prod['coautoria_aluno'] = df_prod['autores'].apply(
            lambda autores: tem_coautoria_aluno(autores, variacoes_alunos)
        )
    else:
        df_prod['coautoria_aluno'] = False

print(f"Periódicos com coautoria de aluno:   {int(df_artigos_final['coautoria_aluno'].sum())}")
print(f"Conferências com coautoria de aluno: {int(df_artigos_congresso_final['coautoria_aluno'].sum())}")

Marcando coautoria de alunos em df_artigos_final e df_artigos_congresso_final...
Periódicos com coautoria de aluno:   834
Conferências com coautoria de aluno: 1872


## 9. Persistência Consolidada no DuckDB

Última etapa: criar (se não existir) o schema relacional no arquivo
`pesquisadores.duckdb` e carregar os DataFrames tratados nas tabelas
correspondentes.

**Modelo relacional:**

- `tb_professores` — tabela "mãe", uma linha por professor (chave `id_lattes`).
- `tb_alunos` — uma linha por aluno, com as variações de nome usadas na
  detecção de coautoria (Seção 8).
- `tb_artigo_periodico` — tabela filha de `tb_professores`, um artigo de
  periódico por linha (resultado da Seção 6).
- `tb_artigo_conferencia` — tabela filha de `tb_professores`, um trabalho de
  congresso por linha (resultado da Seção 7).
- `tb_orientacoes` — tabela filha de `tb_professores`, uma orientação por linha
  (resultado da Seção 4).

A carga é feita em modo *replace*: as tabelas filhas são limpas antes da
tabela mãe (para não violar a integridade referencial) e, em seguida, todo o
conteúdo tratado em memória é inserido novamente.

### 9.1 Criação do schema (tabelas, sequências e chaves estrangeiras)

In [ ]:
print("Conectando ao DuckDB e criando o schema (se ainda não existir)...")
con = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

# --- Tabela mãe: Professores ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR
);
""")

# --- Tabela de Alunos ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_alunos (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    nome_completo_normalizado VARCHAR,
    nome_citacoes_normalizadas VARCHAR,
    variacoes_coautoria VARCHAR
);
""")

# --- Sequências para os IDs automáticos das tabelas filhas ---
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_periodico;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_conferencia;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_orientacao;")

# --- Tabela Filha 1: Artigos de Periódico (cruzados com Scopus) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha 2: Artigos de Conferência (cruzados com a base de eventos) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha 3: Orientações ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# Garante que bancos criados em uma versão anterior do schema (sem estas
# colunas) sejam atualizados ao reexecutar o notebook. Cada ALTER é
# protegido por try/except porque o DuckDB ainda não suporta
# "ADD COLUMN IF NOT EXISTS" de forma totalmente idempotente em todas as versões.
for alter_sql in [
    "ALTER TABLE tb_artigo_periodico ADD COLUMN autores VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN doi VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_alunos ADD COLUMN nome_completo_normalizado VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN nome_citacoes_normalizadas VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN variacoes_coautoria VARCHAR",
]:
    try:
        con.execute(alter_sql)
    except Exception:
        pass  # Coluna já existe — nada a fazer

print("Schema pronto (tabelas criadas ou já existentes).")

Conectando ao DuckDB e criando o schema (se ainda não existir)...
Schema pronto (tabelas criadas ou já existentes).


### 9.2 Carga dos dados (limpeza das tabelas antigas + inserção)

In [ ]:
print("Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...")
# A ordem importa: as tabelas filhas têm FOREIGN KEY para tb_professores,
# então precisam ser esvaziadas antes da tabela mãe.
con.execute("DELETE FROM tb_artigo_periodico")
con.execute("DELETE FROM tb_artigo_conferencia")
con.execute("DELETE FROM tb_orientacoes")
con.execute("DELETE FROM tb_alunos")
con.execute("DELETE FROM tb_professores")

print("Inserindo os dados tratados...")

# --- Tabela Mãe: Professores ---
if not df_pessoas.empty:
    con.execute("INSERT INTO tb_professores SELECT * FROM df_pessoas")

# --- Tabela de Alunos ---
if not df_alunos.empty:
    con.execute("""
        INSERT INTO tb_alunos (
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        FROM df_alunos
    """)

# --- Tabela Filha 1: Artigos de Periódico ---
if not df_artigos_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area
        FROM df_artigos_final
    """)

# --- Tabela Filha 2: Artigos de Conferência ---
if not df_artigos_congresso_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno
        FROM df_artigos_congresso_final
    """)

# --- Tabela Filha 3: Orientações ---
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()

print(f"Processo finalizado! Banco '{ARQUIVO_DUCKDB_DESTINO}' atualizado com o schema completo.")

Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...
Inserindo os dados tratados...
Processo finalizado! Banco 'pesquisadores_teste.duckdb' atualizado com o schema completo.
